# 03 — Multi-Agent: Supervisor Pattern

**Supervisor** = one LLM sees everything and routes to workers. Workers report back.

```
         ┌──────────────┐
  START → │  Supervisor  │ → END
         └──────┬───────┘
                │ routes via Command(goto=...)
         ┌──────┴──────┐
         ▼             ▼
    [ingestion]    [quality]
         │             │
         └──── back to supervisor ────┘
```

Key: `Command` object — lets a node both **update state** and **control routing**.

In [ ]:
from typing import Literal
from typing_extensions import TypedDict, Annotated
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.messages import AIMessage

llm = ChatOllama(model="llama3.2:3b")

## Step 1 — Worker tools

In [ ]:
@tool
def read_from_s3(date: str) -> str:
    """Read raw data from S3 for a given date."""
    return f"Read 50,000 rows from s3://bucket/input/{date}/data.parquet"

@tool
def write_to_redshift(table: str, row_count: int) -> str:
    """Write processed data to Redshift."""
    return f"Wrote {row_count} rows to {table}"

@tool
def run_dq_check(table: str) -> str:
    """Run data quality checks on a table."""
    return f"DQ check on {table}: 0 nulls, 0 duplicates, freshness OK"

@tool
def alert_slack(message: str) -> str:
    """Send an alert to Slack."""
    return f"Alert sent: {message}"

print("Tools defined")

## Step 2 — Worker agents

In [ ]:
ingestion_agent = create_react_agent(
    model=llm,
    tools=[read_from_s3, write_to_redshift],
    prompt="You are a data ingestion agent. Read from S3 and write to Redshift.",
)

quality_agent = create_react_agent(
    model=llm,
    tools=[run_dq_check, alert_slack],
    prompt="You are a data quality agent. Run DQ checks and alert on failures.",
)

print("Worker agents created")

## Step 3 — Worker nodes (use Command to route back)

In [ ]:
def ingestion_node(state: MessagesState) -> Command[Literal["supervisor"]]:
    result = ingestion_agent.invoke(state)
    return Command(
        update={"messages": [AIMessage(content=result["messages"][-1].content, name="ingestion")]},
        goto="supervisor",
    )

def quality_node(state: MessagesState) -> Command[Literal["supervisor"]]:
    result = quality_agent.invoke(state)
    return Command(
        update={"messages": [AIMessage(content=result["messages"][-1].content, name="quality")]},
        goto="supervisor",
    )

print("Worker nodes defined")

## Step 4 — Supervisor node (structured routing)

In [ ]:
class Router(TypedDict):
    next: Annotated[Literal["ingestion", "quality", "FINISH"], "which worker to call next"]
    reasoning: str

def supervisor_node(state: MessagesState) -> Command[Literal["ingestion", "quality", "__end__"]]:
    system = """You are a DE pipeline supervisor. Route to:
- ingestion: to load data from S3 to Redshift
- quality: to run data quality checks
- FINISH: when the pipeline is fully complete

Look at conversation history to see what's already been done."""

    response = llm.with_structured_output(Router).invoke(
        [{"role": "system", "content": system}] + state["messages"]
    )
    goto = END if response["next"] == "FINISH" else response["next"]
    print(f"[supervisor] → {response['next']} | {response['reasoning']}")
    return Command(goto=goto)

print("Supervisor defined")

## Step 5 — Wire the graph

In [ ]:
builder = StateGraph(MessagesState)
builder.add_edge(START, "supervisor")
builder.add_node("supervisor", supervisor_node)
builder.add_node("ingestion",  ingestion_node)
builder.add_node("quality",    quality_node)

graph = builder.compile(checkpointer=InMemorySaver())
print("Graph compiled")

In [ ]:
config = {"configurable": {"thread_id": "pipeline-run-1"}}

result = graph.invoke(
    {"messages": [{"role": "user", "content": "Ingest data for 2024-01-15 then run quality checks"}]},
    config,
)
print("\n=== Final answer ===")
print(result["messages"][-1].content)

## Supervisor vs Handoff — interview answer

> *"Supervisor is centralised control — one LLM sees everything and decides routing. Easier to debug, single point of failure. Handoff is peer-to-peer — agents decide themselves when to transfer. More flexible but harder to trace. For DE pipelines I'd use supervisor because auditability matters more than flexibility — same reason I'd choose Step Functions over EventBridge for complex orchestration."*